In [4]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
# from helper import load_env
# load_env()
from pydantic import BaseModel, Field
from typing import List, Dict, Type
from typing import List, Optional
import os
import yaml

In [5]:
import os, json, time, gc
import logging 

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, Image, Video
from tqdm import tqdm
from openai import OpenAI, AsyncOpenAI
from openai.types.chat import (ChatCompletion, 
                               ChatCompletionChunk,
                               ChatCompletionContentPartTextParam, 
                               ChatCompletionContentPartImageParam,
                               ChatCompletionStreamOptionsParam)
import asyncio
import aiohttp
import pandas as pd
import re


import base64
from PIL import Image
import io

#fix bug with aysncio and jupyter
import nest_asyncio # for langchain async 
nest_asyncio.apply()

In [6]:
import litellm
from litellm import acompletion, completion

### Test LM Studio Connection By Openai API

In [7]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
api_key= "lm-studio"

In [8]:
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=api_key)

# Replace with the exact model name running in LM Studio
model_name = "google/gemma-4-12b" 

In [9]:
ret = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful AI coding assistant."},
        {"role": "user", "content": "Explain how to check memory usage in a Jupyter notebook."}
    ],
    temperature=0.7,
)

Markdown(ret.choices[0].message.content)



Here are the most effective ways to check and monitor memory usage in a Jupyter Notebook, ranging from quick system checks to detailed code profiling.

### 🔹 1. Check Current Process & System Memory (`psutil`)
Best for: **Quickly seeing how much RAM your notebook kernel is using vs. what's available on the machine.**

```python
# Install if needed: pip install psutil
import psutil

# Notebook kernel memory (Resident Set Size)
process = psutil.Process()
mem = process.memory_info()
print(f"Kernel Memory Usage (RSS): {mem.rss / 1024**2:.2f} MB")

# System-wide memory
vm = psutil.virtual_memory()
print(f"System: {vm.percent}% used | Available: {vm.available / 1024**3:.2f} GB")
```

---

### 🔹 2. Profile Specific Cells or Functions (`memory_profiler`)
Best for: **Measuring peak memory usage and memory increment of specific code blocks.**

```python
# Install if needed: pip install memory_profiler
%load_ext memory_profiler

%%memit
large_data = [i**2 for i in range(5_000_000)]
```
**Output example:**
```
peak memory: 384.12 MiB, increment: 186.45 MiB
```
- `peak memory`: Maximum RAM used during execution
- `increment`: Additional RAM consumed by this cell (useful for tracking growth)

> 💡 Tip: Use `%memit expression` for single-line profiling, or `%%memit` for entire cells.

---

### 🔹 3. Track Python Object Allocations (`tracemalloc`)
Best for: **Finding exactly which lines/objects are consuming the most memory (built-in, no install required).**

```python
import tracemalloc

tracemalloc.start()

# --- Your code here ---
data = {"key": [i for i in range(1_000_000)]}
df = __import__('pandas').read_csv("large_file.csv")  # Example
# ----------------------

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')[:10]

print("[ Top 10 Memory Allocations ]")
for stat in top_stats:
    print(stat)

tracemalloc.stop()
```
This shows file paths, line numbers, and memory sizes for the largest allocations. Extremely useful for debugging leaks or inefficient data structures.

---

### 🔹 4. Quick OS-Level Checks (Shell Magic)
Best for: **Zero-install, quick system checks.**

| OS | Command |
|----|---------|
| Linux | `!free -h` or `!cat /proc/meminfo \| grep MemAvailable` |
| macOS | `!vm_stat` (outputs in pages; multiply by 4096/1024³ for GB) |
| Windows | `!wmic process where name="python.exe" get WorkingSetSize` |

---

### ⚠️ Jupyter-Specific Memory Gotchas
Jupyter behaves differently than standard Python scripts:
1. **Outputs accumulate**: Every cell execution stores results in memory (`Out[1]`, `Out[2]`, etc.). Large DataFrames or plots will stay loaded until cleared.
   - Clear specific outputs: `%xdel variable_name` (requires `ipython`) or manually delete & run `import gc; gc.collect()`
   - Reset kernel: `Kernel > Restart & Clear Output`
2. **Garbage collection isn't automatic**: Python's GC runs lazily. Force it after deleting large objects:
   ```python
   del large_df
   import gc
   gc.collect()
   ```
3. **Variables persist across cells**: Unlike scripts, Jupyter keeps the entire namespace alive. Monitor growth over time if your notebook runs for hours.

---

### ✅ When to Use Which?
| Goal | Recommended Tool |
|------------------------|
| Quick kernel/system check | `psutil` |
| Measure cell/function memory impact | `%memit` / `%%memit` |
| Find exact lines/objects causing high usage | `tracemalloc` |
| Debug memory leaks over time | Combine `psutil` + manual tracking or use `%timeit`+`%memit` in loops |

Let me know your specific use case (e.g., pandas DataFrames, machine learning models, long-running simulations) and I can tailor the approach!

## Test LM studio LLM Connection by liteLLM API

In [10]:
# Markdown(completion.choices[0].message.content)

In [11]:
# 3. Define the async function
async def get_chat_completion(
    api_base,
    api_key, 
    model_name = "openai/local-model",
    system_prompt= "You are a helpful assistant.",
    user_prompt="",
    temperature= 0.7,
    max_tokens=4096):
    
    response = await acompletion(
        model=model_name,  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
        api_base=api_base,
        api_key=api_key,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response

In [12]:
%%time
# 4. Execute the async function directly in Jupyter
response = asyncio.run(get_chat_completion(api_base=LM_STUDIO_BASE_URL, 
                                           api_key=api_key,
                                           model_name="openai/local-model",
                                            user_prompt="What is LLM?"))



Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x73545ae6b440> is already entered
Task was destroyed but it is pending!
task: <Task pending name='Task-32' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/aifather/venv-gemma4/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-33' coro=<Kernel.shell_main() running at /home/aifather/venv-gemma4/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/aifather/venv-gemma4/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is pending!
task: <Task pending name='Task-33' coro=<Kernel.shell_main() running at /home/aifather/venv-gemma4

CPU times: user 62.3 ms, sys: 12.6 ms, total: 74.9 ms
Wall time: 28.9 s


In [13]:
Markdown(response.choices[0].message.content)



**LLM** stands for **Large Language Model**. It's a type of artificial intelligence designed to understand, generate, and interact with human language in a highly natural way.

### 🔹 How They Work (Simplified)
- LLMs are built using deep learning architectures, most notably the **transformer** design.
- They're trained on massive datasets containing books, articles, websites, code, conversations, and more.
- Through this training, they learn statistical patterns in language: grammar, facts, reasoning styles, tone, and context.
- When you give them a prompt, they predict the most likely next words based on those learned patterns, producing coherent, human-like text.

### 🔹 Key Capabilities
- Answer questions & explain complex topics
- Write essays, emails, stories, scripts, or code
- Translate languages & summarize long documents
- Engage in multi-turn conversations
- Assist with research, brainstorming, debugging, and analysis

### 🔹 Well-Known Examples
OpenAI's GPT series, Anthropic's Claude, Google's Gemini, Meta's Llama, Mistral, and others. Many are available via APIs, web interfaces, or open-source platforms.

### ⚠️ Important Context & Limitations
- **Pattern matchers, not thinkers**: LLMs don't "understand" meaning like humans do; they're highly advanced statistical predictors.
- **Hallucinations**: They can confidently generate incorrect or fabricated information.
- **Bias & safety**: Training data reflects real-world biases, so outputs can be skewed or inappropriate without proper safeguards.
- **Resource-intensive**: Training and running large models requires significant computing power and energy.

### 💡 Best Practices for Use
- Treat LLM output as a draft or assistant, not a final authority.
- Verify facts, especially for academic, medical, legal, or financial use.
- Prompt clearly and iterate; specify tone, format, and constraints when possible.

Let me know if you'd like a deeper dive into how they're trained, their architecture, ethical considerations, or practical tips for using them!

## Generate COT Data

In [14]:
def generate_cot_data(prompt: str, answer: str) -> str:
    system_prompt = """You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.

Think step by step inside <think> </think> tags.
Focus only on explaining how to discover the hidden rule.
Do NOT output the final answer yourself."""

    user_message = f"""Puzzle:
{prompt}

Correct Answer: {answer}

Please think step by step inside <think> tags about how to discover the transformation rule."""

    try:
        response = asyncio.run (acompletion(
            model="openai/local-model",  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
            api_base=LM_STUDIO_BASE_URL,
            api_key=api_key,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.3,
            max_tokens=1600,
            timeout=180
            # reasoning_effort="medium"
        ))
        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", "") or ""
        content = message.content or ""

        # Combine reasoning
        if reasoning:
            thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
        else:
            thinking_part = f"<think>\n{content.strip()}\n</think>"

        # === Hardcode the final answer (Most Reliable) ===
        final_output = f"{thinking_part}\n\\boxed{{{answer}}}"

        return final_output
        
    except Exception as e:
        print(f"Error generating CoT for prompt: {e}")
        # Fallback: still return something usable
        return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"
                            

In [30]:
testFile ="../src/Dataset/test.csv"
trainFile = "../src/Dataset/train.csv"
cotFile = "train_cot.csv"


In [ ]:
# trainDF = pd.read_csv(trainFile)
# trainDF

In [ ]:
# # Add new column for CoT reasoning
# if "cot_reasoning" not in trainDF.columns:
#     trainDF["cot_reasoning"] = ""

In [ ]:
# trainDF

In [31]:
outputFile = "train_cot2.csv"               # output file with CoT
# MODEL = "gpt-4o"                         # or "claude-3-5-sonnet-20241022"
DELAY = 0.1                                # seconds between API calls (adjust based on rate limit)

In [33]:
print("Checking for existing train_cot.csv...")

if os.path.exists(outputFile):
    print("Found existing train_cot.csv → Resuming...")
    trainDF = pd.read_csv(outputFile)
else:
    print("No existing file found. Starting from train.csv...")
    trainDF = pd.read_csv(trainFile)
    if "cot_reasoning" not in trainDF.columns:
        trainDF["cot_reasoning"] = ""

# Count how many rows still need processing
remaining = trainDF["cot_reasoning"].isna().sum() + (trainDF["cot_reasoning"] == "").sum()
print(f"Total rows: {len(trainDF)}")
print(f"Rows already processed: {len(trainDF) - remaining}")
print(f"Rows left to process: {remaining}\n")

Checking for existing train_cot.csv...
Found existing train_cot.csv → Resuming...
Total rows: 9500
Rows already processed: 50
Rows left to process: 9450



In [ ]:
trainDF

In [ ]:
%%time
# print("Generating Chain-of-Thought data...")

# for idx in tqdm(range(len(trainDF))):
#     if trainDF.loc[idx, "cot_reasoning"]:   # Skip if already generated
#         continue

#     prompt = trainDF.loc[idx, "prompt"]
#     answer = str(trainDF.loc[idx, "answer"]).strip()

#     cot = generate_cot_data(prompt, answer)
#     trainDF.loc[idx, "cot_reasoning"] = cot

#     # Save progress every 50 rows (in case of crash)
#     if (idx + 1) % 20 == 0:
#         trainDF.to_csv(outputFile, index=False)
#         print(f"Saved progress at row {idx + 1}")

#     time.sleep(DELAY)   # Respect API rate limit

if remaining == 0:
    print("✅ All rows already have CoT reasoning. Nothing to do.")
else:
    print("Starting CoT generation (resume mode)...\n")

    processed_count = 0

    for idx in tqdm(range(len(trainDF))):
        current_cot = trainDF.loc[idx, "cot_reasoning"]

        # Skip if already has content
        if pd.notna(current_cot) and str(current_cot).strip() != "":
            continue

        prompt = trainDF.loc[idx, "prompt"]
        answer = str(trainDF.loc[idx, "answer"]).strip()

        cot = generate_cot_data(prompt, answer)
        trainDF.loc[idx, "cot_reasoning"] = cot
        processed_count += 1

        # Save progress every 50 new rows
        if processed_count % 20 == 0:
            trainDF.to_csv(outputFile, index=False)
            print(f"Saved progress. Processed {processed_count} new rows so far.")

        time.sleep(DELAY)

    # Final save
    trainDF.to_csv(outputFile, index=False)
    print(f"\n✅ Finished! Processed {processed_count} new rows.")
    print(f"File saved to: {outputFile}")

Starting CoT generation (resume mode)...



  1%|▎                                     | 70/9500 [13:01<99:18:55, 37.91s/it]

Saved progress. Processed 20 new rows so far.


  1%|▎                                    | 74/9500 [15:34<100:01:51, 38.20s/it]Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x73545ae6b440> is already entered
IOStream.flush timed out
Task was destroyed but it is pending!
task: <Task pending name='Task-704' coro=<_async_in_context.<locals>.run_in_context() done, defined at /home/aifather/venv-gemma4/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-705' coro=<Kernel.shell_main() running at /home/aifather/venv-gemma4/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/aifather/venv-gemma4/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
Task was destroyed but it is 

Saved progress. Processed 40 new rows so far.


...

Saved progress. Processed 680 new rows so far.


  8%|██▊                                | 750/9500 [7:29:57<92:54:54, 38.23s/it]

Saved progress. Processed 700 new rows so far.


  8%|██▊                                | 770/9500 [7:42:41<92:07:56, 37.99s/it]

Saved progress. Processed 720 new rows so far.
